In [1]:
import pandas as pd
import numpy as np
import emoji
import string
import unicodedata
import nltk
import re

In [2]:
df_novo = pd.read_csv('df_novo.csv', encoding = 'utf-8', sep = ';')

portuguese_stopwords = nltk.corpus.stopwords.words('portuguese')

def preprocess_text(text, 
                    remove_stop = True, 
                    stem_words = False, 
                    remove_mentions_hashtags = True):
    """
    eg:
    input: preprocess_text("@water #dream hi hello where are you going be there tomorrow happening happen happens",  
    stem_words = True) 
    output: ['tomorrow', 'happen', 'go', 'hello']
    """

    # Remove emojis
    emoji_pattern = re.compile("[" "\U0001F1E0-\U0001F6FF" "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r"", text)
    text = "".join([x for x in text if x not in emoji.UNICODE_EMOJI])
    
    # Corrige o bud que elimina parte das palavras com acentos
    text = ''.join(ch for ch in unicodedata.normalize('NFKD', text) 
    if not unicodedata.combining(ch))

    if remove_mentions_hashtags:
        text = re.sub(r"@(\w+)", " ", text)
        text = re.sub(r"#(\w+)", " ", text)

    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    regex = re.compile('[' + re.escape(string.punctuation) + '0-9\\r\\t\\n]')
    nopunct = regex.sub(" ", text.lower())
    words = (''.join(nopunct)).split()
    
    

    if(remove_stop):
        words = [w for w in words if w not in portuguese_stopwords]
        words = [w for w in words if len(w) > 2]  

    if(stem_words):
        stemmer = PorterStemmer()
        words = [stemmer.stem(w) for w in words]

    return list(words)
 

# carrega o dataset apenas com as colunas desejadas
df_bag = df_novo[['KEY','COMMENT_ID', 'COMMENT_TEXT']]

df_bag = df_bag.dropna().reset_index().drop(columns=['index'])

# cria a coluna com textos vetorizados
rows, cols = df_bag.shape

df_bag['WORD'] = [preprocess_text(df_bag["COMMENT_TEXT"][row]) for row in range(rows)]

lst_col = 'WORD'

df_bag = pd.DataFrame({col:np.repeat(df_bag[col].values, df_bag[lst_col].str.len())
    for col in df_bag.columns.difference([lst_col])}).assign(**{lst_col:np.concatenate(df_bag[lst_col].values)}
                                                            )[df_bag.columns.tolist()]

C:\ProgramData\Anaconda3\lib\site-packages\IPython\core\interactiveshell.py:3071: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  has_raised = await self.run_ast_nodes(code_ast.body, cell_name,


In [3]:
df_bag

,KEY,COMMENT_ID,COMMENT_TEXT,WORD
0,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,8925759.0,Ótimo,otimo
1,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,8900451.0,Ótimo,otimo
2,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,8621343.0,Ótimo,otimo
3,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,8923399.0,Ótimo,otimo
4,aHR0cHM6Ly93d3cuY2FzYXNiYWhpYS5jb20uYnIvc21hcn...,8787471.0,Ótimo,otimo
...,...,...,...,...
1963585,aHR0cHM6Ly93d3cuYW1hem9uLmNvbS5ici9TbWFydHBob2...,R3JEA1F95PDHZ9,Muito bom msm! O melhor celular custo benefício,celular
1963586,aHR0cHM6Ly93d3cuYW1hem9uLmNvbS5ici9TbWFydHBob2...,R3JEA1F95PDHZ9,Muito bom msm! O melhor celular custo benefício,custo
1963587,aHR0cHM6Ly93d3cuYW1hem9uLmNvbS5ici9TbWFydHBob2...,R3JEA1F95PDHZ9,Muito bom msm! O melhor celular custo benefício,beneficio
1963588,aHR0cHM6Ly93d3cuYW1hem9uLmNvbS5ici9TbWFydHBob2...,R2BRQ41N8CR1JZ,Muito bom. Top,bom
